# Base Paper: Forward-Secure Lattice-Based IBE for Internet of Things

**Reference:** *A lattice-based forward secure IBE scheme for Internet of things* (Pure base paper — NO trust model enhancements).

---

## Overview

This notebook implements the **pure base paper** fs-IBE scheme:

- **Identity-based encryption (IBE):** Uses a user's identity string as the public key.
- **Forward security:** Keys are updated over epochs via a binary tree.
- **Lattice-based:** Security relies on LWE/SIS hardness (post-quantum).

**This version does NOT include:**
- ❌ Trust Model (TrustManager, trust scores)
- ❌ Dilithium-3 signatures
- ❌ Query authentication
- ❌ False Trust Acceptance Rate (FTAR)

---
## P1 — Lattice Infrastructure

Implements TrapGen, SamplePre, gadget matrix, binary tree, Setup, and KeyGen.

In [ ]:
import numpy as np
import hashlib
import sys
import types
import time
import csv
import json
from math import ceil, log2

class LatticeParams:
    def __init__(self, n=16, q=3329, sigma=3.2):
        self.n = n
        self.q = q
        self.k = ceil(log2(q))
        self.m = n * self.k
        self.sigma = sigma

def gadget_matrix(n, q):
    k = ceil(log2(q))
    G = np.zeros((n, n * k), dtype=int)
    for i in range(n):
        for j in range(k):
            G[i, i * k + j] = 1 << j
    return G % q

def bit_decompose(vec, q):
    k = ceil(log2(q))
    bits = []
    for v in vec % q:
        for i in range(k):
            bits.append((v >> i) & 1)
    return np.array(bits, dtype=int)

def TrapGen(params):
    n, q, m = params.n, params.q, params.m
    A_bar = np.random.randint(0, q, size=(n, m))
    G = gadget_matrix(n, q)
    A = np.hstack([A_bar, G]) % q
    T_A = np.vstack([np.zeros((m, m), dtype=int), np.eye(m, dtype=int)])
    return A, T_A

def discrete_gaussian(shape, sigma):
    return np.round(np.random.normal(0, sigma, size=shape)).astype(int)

def SamplePre(A, T_A, u, params):
    q = params.q
    _, m2 = A.shape
    m = m2 // 2
    y = bit_decompose(u, q)
    e = np.zeros(2 * m, dtype=int)
    e[m:] = y
    return e % q

def G_vector(data, params):
    vec = []
    ctr = 0
    while len(vec) < params.n:
        h = hashlib.sha256((data + str(ctr)).encode()).digest()
        vec.extend(h[:min(32, params.n - len(vec))])
        ctr += 1
    return np.array(vec[:params.n], dtype=int) % params.q

class BinaryTreeNode:
    def __init__(self, label):
        self.label = label
        self.left = None
        self.right = None

class BinaryTree:
    def __init__(self, depth):
        self.depth = depth
        self.root = self._build(0, 2**depth - 1)
    def _build(self, l, r):
        if l > r:
            return None
        mid = (l + r) // 2
        node = BinaryTreeNode(mid)
        node.left = self._build(l, mid - 1)
        node.right = self._build(mid + 1, r)
        return node

def Setup(tree_depth=4, params=None):
    if params is None:
        params = LatticeParams()
    A, T_A = TrapGen(params)
    return {"params": params, "A": A, "T_A": T_A, "tree": BinaryTree(tree_depth)}

def KeyGen(system, user_id):
    params = system["params"]
    u = G_vector(user_id, params)
    return SamplePre(system["A"], system["T_A"], u, params)

# Register modules
lattice_infrastructure = types.ModuleType("lattice_infrastructure")
for name in ["LatticeParams", "gadget_matrix", "bit_decompose", "TrapGen", "discrete_gaussian", "SamplePre", "G_vector", "BinaryTreeNode", "BinaryTree", "Setup", "KeyGen"]:
    lattice_infrastructure.__dict__[name] = globals()[name]
sys.modules["LatticeCrypto"] = lattice_infrastructure
sys.modules["lattice_infrastructure"] = lattice_infrastructure
P1 = lattice_infrastructure

# P1 unit test
params = LatticeParams(n=64)
system = Setup(tree_depth=3, params=params)
sk = KeyGen(system, "Alice")
u = G_vector("Alice", params)
assert np.array_equal((system["A"] @ sk) % params.q, u)
print("[P1] Lattice Infrastructure: PASS", flush=True)

---
## P2 — Forward Security

Implements Encrypt/Decrypt (Dual Regev IBE), key evolution via binary tree minimal cover.

In [ ]:
class UserOps:
    def __init__(self, system):
        self.params = system["params"]
        self.A = system["A"]
        self.T_A = system["T_A"]
        self.tree = system["tree"]
        self.total_epochs = 2 ** self.tree.depth

    def get_min_cover(self, current_time):
        cover = []
        def dfs(node, l, r):
            if node is None or r < current_time:
                return
            if l >= current_time:
                cover.append(node.label)
                return
            mid = (l + r) // 2
            dfs(node.left, l, mid - 1)
            if mid >= current_time:
                cover.append(node.label)
            dfs(node.right, mid + 1, r)
        dfs(self.tree.root, 0, self.total_epochs - 1)
        return sorted(set(cover))

    def Update(self, key_bundle, t):
        next_t = t + 1
        if next_t >= self.total_epochs:
            return {}, []
        needed = self.get_min_cover(next_t)
        return {k: key_bundle[k] for k in needed if k in key_bundle}, needed

    def simulate_key_evolution(self, user_id, nodes):
        bundle = {}
        for node in nodes:
            uid = f"{user_id}_{node}"
            u = P1.G_vector(uid, self.params)
            bundle[node] = P1.SamplePre(self.A, self.T_A, u, self.params)
        return bundle

    def Encrypt(self, user_id, epoch, bit):
        uid = f"{user_id}_{epoch}"
        u = P1.G_vector(uid, self.params)
        q = self.params.q
        s = np.random.randint(0, q, size=self.params.n)
        e1 = P1.discrete_gaussian((self.A.shape[1],), self.params.sigma)
        e2 = P1.discrete_gaussian((1,), self.params.sigma)[0]
        c1 = (self.A.T @ s + e1) % q
        c2 = (u @ s + e2 + (q // 2) * bit) % q
        return {"c1": c1, "c2": c2, "epoch": epoch}

    def Decrypt(self, ct, key_bundle):
        sk = key_bundle.get(ct["epoch"])
        if sk is None:
            return None
        q = self.params.q
        approx = (ct["c2"] - sk @ ct["c1"]) % q
        return 0 if min(approx, q - approx) < abs(approx - q // 2) else 1

# Register
forward_security = types.ModuleType("forward_security")
forward_security.UserOps = UserOps
sys.modules["forward_security"] = forward_security

# P2 unit test
system = P1.Setup(tree_depth=3, params=P1.LatticeParams(n=64))
ops = UserOps(system)
keys = ops.simulate_key_evolution("Alice", [0, 1, 2])
ct = ops.Encrypt("Alice", 1, 1)
assert ops.Decrypt(ct, keys) == 1
print("[P2] Forward Security: PASS", flush=True)

---
## Table 1 — fs-IBE Parameter Selection

| Parameter | n | q | δ (approx) | NIST Level | bits security |
|-----------|-----|------|------------|------------|---------------|
| PARA.512 | 512 | 3329 | 2^(-139) | 1 | 143 |
| PARA.768 | 768 | 3329 | 2^(-164) | 3 | 207 |
| PARA.1024 | 1024 | 3329 | 2^(-174) | 5 | 272 |

In [ ]:
FS_IBE_TABLE = [
    {"parameter": "PARA.512", "n": 512, "q": 3329, "delta_approx": "2^(-139)", "nist_level": 1, "bits_security": 143},
    {"parameter": "PARA.768", "n": 768, "q": 3329, "delta_approx": "2^(-164)", "nist_level": 3, "bits_security": 207},
    {"parameter": "PARA.1024", "n": 1024, "q": 3329, "delta_approx": "2^(-174)", "nist_level": 5, "bits_security": 272},
]

def get_lattice_params(parameter_name):
    for row in FS_IBE_TABLE:
        if row["parameter"] == parameter_name:
            return LatticeParams(n=row["n"], q=row["q"])
    raise KeyError(f"Unknown parameter: {parameter_name}")

def print_table_1():
    header = f"{'Parameter':<12} {'n':<8} {'q':<8} {'δ (approx)':<14} {'NIST Level':<12} {'bits security':<16}"
    sep = "-" * 80
    lines = ["Table 1: The parameter selection of our fs-IBE.", "", header, sep]
    for row in FS_IBE_TABLE:
        lines.append(f"{row['parameter']:<12} {row['n']:<8} {row['q']:<8} {row['delta_approx']:<14} {row['nist_level']:<12} {row['bits_security']:<16}")
    print("\n".join(lines), flush=True)

# Register
fs_ibe_params = types.ModuleType("fs_ibe_params")
fs_ibe_params.FS_IBE_TABLE = FS_IBE_TABLE
fs_ibe_params.get_lattice_params = get_lattice_params
sys.modules["fs_ibe_params"] = fs_ibe_params

print_table_1()

---
## P4 — Simulation & Benchmarking (Base Paper — No Trust Model)

Runs the full simulation for all 3 parameter sets. **No trust verification, no FTAR.**

In [ ]:
def run_simulation(n=64, num_data=5, num_queries=10, tree_depth=3, param_name=None):
    params = P1.LatticeParams(n=n)
    system = P1.Setup(tree_depth=tree_depth, params=params)
    ops = UserOps(system)
    user_id = "Alice"
    epoch = 1
    nodes = list(range(min(epoch + 2, 2 ** tree_depth)))
    keys = ops.simulate_key_evolution(user_id, nodes)

    # Data encryption
    t0 = time.perf_counter()
    encrypted_data = []
    for i in range(num_data):
        ct = ops.Encrypt(user_id, epoch, i % 2)
        encrypted_data.append(ct)
    data_encryption_time = time.perf_counter() - t0

    # Query encryption (plain fs-IBE, no signing)
    t_enc_q_list = []
    query_cts = []
    for _ in range(num_queries):
        t0 = time.perf_counter()
        keyword_ct = ops.Encrypt(user_id, epoch, 1)
        t_enc_q_list.append(time.perf_counter() - t0)
        query_cts.append(keyword_ct)
    query_encryption_time = sum(t_enc_q_list) / len(t_enc_q_list) if t_enc_q_list else 0

    # Match
    t_match_list = []
    for qct in query_cts:
        t0 = time.perf_counter()
        matched = [i for i, ct in enumerate(encrypted_data) if ct["epoch"] == qct["epoch"]]
        t_match_list.append(time.perf_counter() - t0)
    match_time = sum(t_match_list) / len(t_match_list) if t_match_list else 0

    # Data decryption
    t0 = time.perf_counter()
    for ct in encrypted_data:
        ops.Decrypt(ct, keys)
    data_decryption_time = time.perf_counter() - t0

    # Decryption per query
    t_dec_list = []
    for qct in query_cts:
        matched = [i for i, ct in enumerate(encrypted_data) if ct["epoch"] == qct["epoch"]]
        t0 = time.perf_counter()
        for idx in matched:
            ops.Decrypt(encrypted_data[idx], keys)
        t_dec_list.append(time.perf_counter() - t0)
    query_decryption_time = sum(t_dec_list) / len(t_dec_list) if t_dec_list else 0

    t_query = query_encryption_time + match_time + query_decryption_time
    total_query_time = t_query * num_queries
    query_throughput = num_queries / total_query_time if total_query_time > 0 else 0
    overall_latency = data_encryption_time + total_query_time + data_decryption_time
    overall_throughput = (num_data + num_queries) / overall_latency if overall_latency > 0 else 0

    out = {
        "data_encryption_time_s": data_encryption_time,
        "query_encryption_time_s": query_encryption_time,
        "data_decryption_time_s": data_decryption_time,
        "query_execution_latency_s": t_query,
        "query_throughput_per_s": query_throughput,
        "overall_model_throughput_per_s": overall_throughput,
        "overall_model_latency_s": overall_latency,
        "false_trust_acceptance_rate": "N/A",
        "num_data": num_data, "num_queries": num_queries,
    }
    if param_name is not None:
        out["parameter"] = param_name
        out["n"] = n
    return out

def print_results(metrics, param_name=None):
    if param_name:
        print("\n" + "=" * 60, flush=True)
        print(f"  Parameter: {param_name}  (n = {metrics.get('n', '—')})", flush=True)
        print("=" * 60, flush=True)
    print(f"  Data encryption time          : {metrics['data_encryption_time_s']:.6f} s", flush=True)
    print(f"  Query encryption time         : {metrics['query_encryption_time_s']:.6f} s", flush=True)
    print(f"  Data decryption time          : {metrics['data_decryption_time_s']:.6f} s", flush=True)
    print(f"  Query execution latency       : {metrics['query_execution_latency_s']:.6f} s", flush=True)
    print(f"  Query throughput              : {metrics['query_throughput_per_s']:.2f} queries/s", flush=True)
    print(f"  Overall model throughput      : {metrics['overall_model_throughput_per_s']:.2f} ops/s", flush=True)
    print(f"  Overall model latency         : {metrics['overall_model_latency_s']:.6f} s", flush=True)
    print(f"  Trust model                   : N/A (base paper)", flush=True)
    print("=" * 60, flush=True)

# Run all 3 parameter sets
all_metrics = []
for row in FS_IBE_TABLE:
    param_name, n = row["parameter"], row["n"]
    print(f"  Running {param_name} (n={n}) ...", flush=True)
    m = run_simulation(n=n, num_data=5, num_queries=10, tree_depth=3, param_name=param_name)
    m["bits_security"] = row["bits_security"]
    m["nist_level"] = row["nist_level"]
    all_metrics.append(m)

for m in all_metrics:
    print_results(m, param_name=m["parameter"])

# Save CSV
fieldnames = ["parameter", "n", "bits_security", "nist_level", "data_encryption_time_s", "query_encryption_time_s", "data_decryption_time_s", "query_execution_latency_s", "query_throughput_per_s", "overall_model_throughput_per_s", "overall_model_latency_s", "false_trust_acceptance_rate", "num_data", "num_queries"]
try:
    with open("Results_Report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for m in all_metrics:
            w.writerow(m)
    print("  Saved: Results_Report.csv", flush=True)
except PermissionError:
    with open("Results_Report_output.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for m in all_metrics:
            w.writerow(m)
    print("  Saved: Results_Report_output.csv (original locked)", flush=True)

---
## Full Pipeline: Message → Encrypt → Decrypt → Result

Enter a message below to encrypt and decrypt it using the base paper fs-IBE scheme.

In [ ]:
print("Enter your message (press Enter when done; leave empty for default 'Hello IoT'):", flush=True)
USER_MESSAGE = input().strip() or "Hello IoT"
print("Processing...", flush=True)

USER_ID = "Alice"
EPOCH = 1
PARAM_SET = "PARA.512"
TREE_DEPTH = 3

def message_to_bits(msg: str) -> list:
    bits = []
    for b in msg.encode("utf-8"):
        for i in range(7, -1, -1):
            bits.append((b >> i) & 1)
    return bits

def bits_to_message(bits: list) -> str:
    n = len(bits)
    if n % 8 != 0:
        n = (n // 8) * 8
    bytes_list = []
    for i in range(0, n, 8):
        byte_val = sum(bits[i + j] << (7 - j) for j in range(8))
        bytes_list.append(byte_val)
    return bytes(bytes_list).decode("utf-8", errors="replace")

params = get_lattice_params(PARAM_SET)
system = P1.Setup(tree_depth=TREE_DEPTH, params=params)
ops = UserOps(system)
nodes = list(range(min(EPOCH + 2, 2 ** TREE_DEPTH)))
key_bundle = ops.simulate_key_evolution(USER_ID, nodes)

bits_original = message_to_bits(USER_MESSAGE)
t0 = time.perf_counter()
ciphertexts = []
for bit in bits_original:
    ct = ops.Encrypt(USER_ID, EPOCH, bit)
    ciphertexts.append(ct)
encrypt_time = time.perf_counter() - t0

t0 = time.perf_counter()
bits_decrypted = []
for ct in ciphertexts:
    b = ops.Decrypt(ct, key_bundle)
    bits_decrypted.append(b if b is not None else 0)
decrypt_time = time.perf_counter() - t0

decrypted_message = bits_to_message(bits_decrypted)

print("\n" + "=" * 70, flush=True)
print("  BASE PAPER — FORWARD-SECURE LATTICE-BASED IBE — FULL PIPELINE", flush=True)
print("=" * 70, flush=True)
print(f"\n  1. ORIGINAL MESSAGE: \"{USER_MESSAGE}\"", flush=True)
print(f"  2. BITS: {len(bits_original)} bits", flush=True)
print(f"  3. CIPHERTEXTS: {len(ciphertexts)} (one per bit, epoch={EPOCH})", flush=True)
print(f"  4. DECRYPTED MESSAGE: \"{decrypted_message}\"", flush=True)
print(f"  5. PARAMS: {PARAM_SET}, n={params.n}, q={params.q}, tree_depth={TREE_DEPTH}", flush=True)
print(f"  6. TIMINGS: encrypt={encrypt_time:.6f}s, decrypt={decrypt_time:.6f}s", flush=True)
match = USER_MESSAGE == decrypted_message
print(f"  7. VERIFICATION: {'PASS ✓' if match else 'MISMATCH ✗'}", flush=True)
print("\n" + "=" * 70, flush=True)

---
## Summary

This notebook implements the **pure base paper** fs-IBE scheme. It provides:

- P1: Lattice Infrastructure (TrapGen, SamplePre, BinaryTree)
- P2: Forward Security (Encrypt, Decrypt, Key Update)
- Table 1: PARA.512, PARA.768, PARA.1024 parameter sets
- P4: Simulation with benchmarks (Results_Report.csv)
- Full Pipeline: Message → Encrypt → Decrypt → Verify

**For the enhanced version with trust model**, see the `Forward-Secure-Lattice-Based-Encryption-For-Internet-Of-Things` folder.